In [ ]:
import pandas as pd

tabs = ["withdrawal_requests", "account_snapshot",
        "pending_activity", "destination_registry"]

for tab in tabs:
    df = pd.read_excel("withdrawals.xlsx", sheet_name=tab)
    print(f"\n--- {tab} ---")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print(df.head(5))


--- withdrawal_requests ---
Shape: (128, 11)
Columns: ['request_id', 'created_at', 'client_id', 'account_id', 'amount', 'currency', 'destination_type', 'destination_id', 'requested_speed', 'channel', 'note']
  request_id                        created_at client_id account_id    amount  \
0    W100055  2026-03-06T01:00:13.581307+00:00      C007      A0032   2000.27   
1    W100040  2026-03-06T04:27:02.907998+00:00      C021      A0008   4048.65   
2    W100019  2026-03-06T01:37:44.488632+00:00      C023      A0014   3352.79   
3    W100031  2026-03-05T15:24:11.787265+00:00      C013      A0027  26978.71   
4    W100098  2026-03-05T11:47:49.487155+00:00      C006      A0023   2861.14   

  currency destination_type destination_id requested_speed channel note  
0      USD           crypto         D00012        standard     app  NaN  
1      USD             bank         D00037        standard     app  NaN  
2      USD             bank         D00039        standard     ops  NaN  
3      U

In [ ]:
import pandas as pd
import numpy as np
from datetime import timedelta

In [ ]:
BUFFER_USD       = 50
RECENT_DEST_DAYS = 7
DUP_WINDOW_MIN   = 15

SEVERITY = {
    "KYC_NOT_VERIFIED"                    : 100,
    "ACCOUNT_NOT_ACTIVE"                  : 95,
    "UNWHITELISTED_HIGH_AML"              : 90,
    "INVALID_AMOUNT"                      : 85,
    "DUPLICATE_REQUEST"                   : 70,
    "INSUFFICIENT_SETTLED_AFTER_BUFFER"   : 65,
    "INSUFFICIENT_AVAILABLE_AFTER_BUFFER" : 55,
    "DEST_CHANGED_RECENTLY"               : 45,
    "URGENT_RISK_TIER"                    : 35,
}

In [ ]:
FILE = "withdrawals.xlsx"

requests  = pd.read_excel(FILE, sheet_name="withdrawal_requests")
snapshot  = pd.read_excel(FILE, sheet_name="account_snapshot")
pending   = pd.read_excel(FILE, sheet_name="pending_activity")   # loaded, not used in rules
registry  = pd.read_excel(FILE, sheet_name="destination_registry")

# Normalize all timestamps to UTC
for df, cols in [
    (requests, ["created_at"]),
    (snapshot,  ["as_of"]),
    (registry,  ["last_changed_at"]),
]:
    for col in cols:
        df[col] = pd.to_datetime(df[col], utc=True)

# Merge requests with account snapshot
df = requests.merge(snapshot, on="account_id", how="left", suffixes=("", "_snap"))

# Merge with destination registry
df = df.merge(
    registry[["destination_id", "is_whitelisted", "last_changed_at"]],
    on="destination_id",
    how="left"
)

# Fill missing values for safety (Point 5 & 6)
df["account_status"]  = df["account_status"].fillna("unknown")
df["kyc_status"]      = df["kyc_status"].fillna("unknown")
df["aml_risk_tier"]   = df["aml_risk_tier"].fillna("unknown")
df["is_whitelisted"]  = df["is_whitelisted"].fillna(False)
df["available_cash"]  = df["available_cash"].fillna(0)
df["settled_cash"]    = df["settled_cash"].fillna(0)

print(f"Total requests loaded: {len(df)}")

Total requests loaded: 128


In [ ]:
# Sort by created_at to ensure direction is correct (Point 7)
df = df.sort_values("created_at").reset_index(drop=True)

dup_window = timedelta(minutes=DUP_WINDOW_MIN)
is_duplicate = pd.Series(False, index=df.index)

for i, row in df.iterrows():
    prior = df[
        (df["account_id"]      == row["account_id"])     &
        (df["amount"]          == row["amount"])          &
        (df["destination_id"]  == row["destination_id"]) &
        (df["created_at"]      <  row["created_at"])      &
        (df["created_at"]      >= row["created_at"] - dup_window)
    ]
    if not prior.empty:
        is_duplicate.at[i] = True

df["is_duplicate"] = is_duplicate

In [ ]:
def evaluate_request(row):
    reject_reasons = []
    hold_reasons   = []

    # --- REJECT CONDITIONS ---
    if row["kyc_status"] != "verified":
        reject_reasons.append("KYC_NOT_VERIFIED")

    if row["account_status"] != "active":
        reject_reasons.append("ACCOUNT_NOT_ACTIVE")

    if row["aml_risk_tier"] == "high" and row["is_whitelisted"] == False:
        reject_reasons.append("UNWHITELISTED_HIGH_AML")

    if row["amount"] <= 0:
        reject_reasons.append("INVALID_AMOUNT")

    if row["is_duplicate"]:
        reject_reasons.append("DUPLICATE_REQUEST")

    # If any reject reason found, return immediately
    if reject_reasons:
        severity = max(SEVERITY[r] for r in reject_reasons)
        return "REJECT", reject_reasons, severity

    # --- HOLD CONDITIONS ---
    if row["settled_cash"] - row["amount"] < BUFFER_USD:
        hold_reasons.append("INSUFFICIENT_SETTLED_AFTER_BUFFER")

    if row["available_cash"] - row["amount"] < BUFFER_USD:
        hold_reasons.append("INSUFFICIENT_AVAILABLE_AFTER_BUFFER")

    # Compare last_changed_at vs as_of (Point 2 — correct reference date)
    if pd.notna(row["last_changed_at"]) and pd.notna(row["as_of"]):
        days_since_change = (row["as_of"] - row["last_changed_at"]).days
        if days_since_change <= RECENT_DEST_DAYS:
            hold_reasons.append("DEST_CHANGED_RECENTLY")

    if row["requested_speed"] == "urgent" and row["aml_risk_tier"] in ("medium", "high"):
        hold_reasons.append("URGENT_RISK_TIER")

    if hold_reasons:
        severity = max(SEVERITY[r] for r in hold_reasons)
        return "HOLD", hold_reasons, severity

    # --- APPROVE ---
    return "APPROVE", [], 0


# Apply engine
results = df.apply(evaluate_request, axis=1, result_type="expand")
results.columns = ["decision", "reason_codes", "severity"]
df = pd.concat([df, results], axis=1)

In [ ]:
# Output 1 — Full decision database
decisions_output = df[[
    "request_id", "account_id", "amount",
    "decision", "reason_codes", "severity"
]].copy()

decisions_output["reason_codes"] = decisions_output["reason_codes"].apply(
    lambda x: ", ".join(x) if x else "NONE"
)

print("=== Decision Summary ===")
print(decisions_output["decision"].value_counts())
print()
print(decisions_output)

=== Decision Summary ===
decision
APPROVE    55
HOLD       38
REJECT     35
Name: count, dtype: int64

    request_id account_id    amount decision  \
0      W100101      A0017   1735.94     HOLD   
1      W100029      A0003  27278.01     HOLD   
2      W100109      A0028   7110.68     HOLD   
3      W100035      A0020   3445.06  APPROVE   
4      W100105      A0022    992.26   REJECT   
..         ...        ...       ...      ...   
123    W100018      A0010   1255.85  APPROVE   
124    W100002      A0008   4505.68  APPROVE   
125    W100046      A0023   8218.61     HOLD   
126    W100107      A0026  19245.96     HOLD   
127    W100062      A0004   4146.56  APPROVE   

                                          reason_codes  severity  
0                                DEST_CHANGED_RECENTLY        45  
1    INSUFFICIENT_SETTLED_AFTER_BUFFER, INSUFFICIEN...        65  
2    INSUFFICIENT_SETTLED_AFTER_BUFFER, INSUFFICIEN...        65  
3                                                 NO

In [11]:
# Output 2 — Review file (HOLD only, sorted by severity desc)
review_file = decisions_output[
    decisions_output["decision"] == "HOLD"
].sort_values("severity", ascending=False).reset_index(drop=True)

print(f"\n=== Review File — {len(review_file)} cases ===")
print(review_file)


=== Review File — 38 cases ===
   request_id account_id    amount decision  \
0     W100029      A0003  27278.01     HOLD   
1     W100109      A0028   7110.68     HOLD   
2     W100054      A0002   6719.15     HOLD   
3     W100045      A0030  27013.79     HOLD   
4     W100022      A0019  11612.55     HOLD   
5     W100052      A0004   9835.49     HOLD   
6     W100005      A0028   6859.80     HOLD   
7     W100072      A0020   8669.22     HOLD   
8     W100058      A0018  10730.44     HOLD   
9     W100012      A0035  20942.37     HOLD   
10    W100069      A0020   8906.77     HOLD   
11    W100032      A0010   9330.97     HOLD   
12    W100007      A0018    673.76     HOLD   
13    W100101      A0017   1735.94     HOLD   
14    W100004      A0026  21878.31     HOLD   
15    W100044      A0003  10244.98     HOLD   
16    W100016      A0018   2425.21     HOLD   
17    W100037      A0021    942.20     HOLD   
18    W100103      A0018   4144.94     HOLD   
19    W100110      A0011   6

In [12]:
# Export to Excel
decisions_output.to_excel("decisions_output.xlsx", index=False)
review_file.to_excel("review_file.xlsx", index=False)

print("Files exported: decisions_output.xlsx | review_file.xlsx")

Files exported: decisions_output.xlsx | review_file.xlsx


In [13]:
# Sanity checks
print("=== Reject breakdown ===")
reject_df = decisions_output[decisions_output["decision"] == "REJECT"]
for _, row in reject_df.iterrows():
    print(f"{row['request_id']} | {row['reason_codes']} | sev {row['severity']}")

=== Reject breakdown ===
W100105 | KYC_NOT_VERIFIED | sev 100
W100070 | ACCOUNT_NOT_ACTIVE | sev 95
W100064_DUP | DUPLICATE_REQUEST | sev 70
W100067 | ACCOUNT_NOT_ACTIVE | sev 95
W100082 | KYC_NOT_VERIFIED | sev 100
W100044_DUP | DUPLICATE_REQUEST | sev 70
W100004_DUP | DUPLICATE_REQUEST | sev 70
W100010_DUP | DUPLICATE_REQUEST | sev 70
W100001 | KYC_NOT_VERIFIED | sev 100
W100017 | ACCOUNT_NOT_ACTIVE | sev 95
W100039 | ACCOUNT_NOT_ACTIVE | sev 95
W100031 | ACCOUNT_NOT_ACTIVE | sev 95
W100073_DUP | DUPLICATE_REQUEST | sev 70
W100068 | INVALID_AMOUNT | sev 85
W100051 | KYC_NOT_VERIFIED | sev 100
W100055_DUP | DUPLICATE_REQUEST | sev 70
W100019 | ACCOUNT_NOT_ACTIVE | sev 95
W100087 | KYC_NOT_VERIFIED | sev 100
W100112 | ACCOUNT_NOT_ACTIVE | sev 95
W100050 | KYC_NOT_VERIFIED | sev 100
W100020 | KYC_NOT_VERIFIED | sev 100
W100063 | ACCOUNT_NOT_ACTIVE | sev 95
W100021 | ACCOUNT_NOT_ACTIVE | sev 95
W100078 | ACCOUNT_NOT_ACTIVE | sev 95
W100089 | ACCOUNT_NOT_ACTIVE | sev 95
W100074 | ACCOUNT_

In [14]:
# Verify high AML + whitelist status
high_aml = df[df["aml_risk_tier"] == "high"][["request_id", "account_id", "aml_risk_tier", "is_whitelisted", "destination_id"]]
print(high_aml)

      request_id account_id aml_risk_tier  is_whitelisted destination_id
3        W100035      A0020          high            True         D00024
9        W100067      A0024          high            True         D00048
14       W100082      A0012          high            True         D00052
22       W100099      A0021          high            True         D00026
26       W100001      A0016          high            True         D00040
27       W100052      A0004          high            True         D00024
28       W100098      A0023          high            True         D00011
34       W100075      A0004          high            True         D00024
41       W100088      A0020          high            True         D00024
42       W100048      A0020          high            True         D00024
44       W100115      A0021          high            True         D00026
47       W100059      A0020          high            True         D00024
50       W100009      A0023          high          

## 3. Motor de Decisiones

### Contexto

Este módulo procesa solicitudes de retiro y determina automáticamente
su estado operativo: APPROVE, HOLD o REJECT. Cada decisión incluye
los reason codes activados y un puntaje de severidad que permite
priorizar la revisión manual de casos en cola.

Las reglas están parametrizadas mediante constantes configurables:

| Parámetro        | Valor | Descripción                                      |
|------------------|-------|--------------------------------------------------|
| BUFFER_USD       | $50   | Balance mínimo requerido post-retiro             |
| RECENT_DEST_DAYS | 7     | Días de gracia tras cambio de destino            |
| DUP_WINDOW_MIN   | 15    | Ventana en minutos para detección de duplicados  |

---

### Lógica de decisión

El motor evalúa cada solicitud en dos capas ordenadas por prioridad:

**Capa 1 — REJECT** (evaluada primero, cualquier match detiene el proceso):
- KYC no verificado
- Cuenta no activa
- Monto inválido (≤ 0)
- Destino de alto riesgo AML no whitelisted
- Solicitud duplicada (mismo account + amount + destination en 15 min)

**Capa 2 — HOLD** (solo si no hay REJECT):
- Cash insuficiente post-buffer (settled o available)
- Destino modificado en los últimos 7 días
- Retiro urgente con perfil de riesgo AML medio o alto

Si ninguna condición se activa, el retiro es **APPROVE** con severidad 0.

---

### Resultados sobre el dataset

| Decisión | Cantidad | % del total |
|----------|----------|-------------|
| APPROVE  | 55       | 43.0%       |
| HOLD     | 38       | 29.7%       |
| REJECT   | 35       | 27.3%       |

**Distribución de REJECTs por razón:**

| Reason Code          | Casos |
|----------------------|-------|
| ACCOUNT_NOT_ACTIVE   | 18    |
| KYC_NOT_VERIFIED     | 7     |
| DUPLICATE_REQUEST    | 8     |
| INVALID_AMOUNT       | 1     |
| UNWHITELISTED_HIGH_AML | 0   |

> Nota: UNWHITELISTED_HIGH_AML no se activó en este dataset porque
> todos los destinos con aml_risk_tier = high tienen is_whitelisted = True.
> La regla está implementada y funcional — simplemente los datos no
> generaron ese escenario.

**HOLDs por razón dominante:**

| Reason Code                          | Casos |
|--------------------------------------|-------|
| INSUFFICIENT_SETTLED_AFTER_BUFFER    | 11    |
| DEST_CHANGED_RECENTLY                | 18    |
| URGENT_RISK_TIER                     | 9     |

El review file exportado contiene los 38 casos HOLD ordenados
por severidad descendente, listos para revisión manual del equipo
de operaciones.

---

### Nota sobre pending_activity

La tabla `pending_activity` fue cargada pero no interviene
directamente en las reglas de decisión definidas en el enunciado.
En un motor de producción, esta tabla sería relevante para calcular
el cash efectivamente disponible considerando operaciones aún no
liquidadas, refinando las condiciones de HOLD por liquidez.

## Simulador de Decisiones — Withdrawal Engine

La lógica de decisión no es solo código: es la primera línea de defensa
operativa entre una solicitud de retiro y el dinero del cliente.

Para ir más allá del motor batch, construimos un simulador interactivo
que permite probar cualquier combinación de parámetros en tiempo real
y ver exactamente qué decisión tomaría el sistema — y por qué.

Esto tiene valor práctico inmediato: el equipo de operaciones puede
recrear un caso específico, entender qué regla lo disparó, y tomar
decisiones informadas sin tener que correr el pipeline completo.

> En un entorno de producción, este simulador evolucionaría hacia
> una interfaz más robusta: integrada con datos en vivo, con historial
> de decisiones por cliente, y diseñada para que un analista de
> operaciones pueda actuar en segundos. La velocidad de reacción
> en wealth management no es un detalle — es parte del servicio.

In [15]:
import ipywidgets as widgets
from IPython.display import display, HTML
from datetime import datetime, timezone

# ── UI Components ────────────────────────────────────────────────
title = widgets.HTML("<h3 style='color:#2c3e50'>🏦 Withdrawal Decision Simulator</h3>")

w_account_status = widgets.Dropdown(
    options=["active", "frozen", "suspended", "closed"],
    value="active", description="Account Status:", style={"description_width": "160px"}
)
w_kyc_status = widgets.Dropdown(
    options=["verified", "pending", "failed"],
    value="verified", description="KYC Status:", style={"description_width": "160px"}
)
w_aml_tier = widgets.Dropdown(
    options=["low", "medium", "high"],
    value="low", description="AML Risk Tier:", style={"description_width": "160px"}
)
w_whitelisted = widgets.Checkbox(
    value=True, description="Is Whitelisted"
)
w_amount = widgets.FloatText(
    value=1000.0, description="Amount (USD):", style={"description_width": "160px"}
)
w_available = widgets.FloatText(
    value=5000.0, description="Available Cash:", style={"description_width": "160px"}
)
w_settled = widgets.FloatText(
    value=4500.0, description="Settled Cash:", style={"description_width": "160px"}
)
w_speed = widgets.Dropdown(
    options=["standard", "urgent"],
    value="standard", description="Speed:", style={"description_width": "160px"}
)
w_days_since_change = widgets.IntSlider(
    value=30, min=0, max=60, step=1,
    description="Days since dest change:", style={"description_width": "160px"},
    layout=widgets.Layout(width="420px")
)
w_is_duplicate = widgets.Checkbox(
    value=False, description="Is Duplicate Request"
)

btn = widgets.Button(
    description="Evaluate Withdrawal",
    button_style="primary",
    layout=widgets.Layout(width="200px", margin="10px 0px")
)

output = widgets.Output()

# ── Decision Engine ──────────────────────────────────────────────
def run_simulation(b):
    output.clear_output()

    reject_reasons = []
    hold_reasons   = []

    if w_kyc_status.value != "verified":
        reject_reasons.append("KYC_NOT_VERIFIED")
    if w_account_status.value != "active":
        reject_reasons.append("ACCOUNT_NOT_ACTIVE")
    if w_aml_tier.value == "high" and not w_whitelisted.value:
        reject_reasons.append("UNWHITELISTED_HIGH_AML")
    if w_amount.value <= 0:
        reject_reasons.append("INVALID_AMOUNT")
    if w_is_duplicate.value:
        reject_reasons.append("DUPLICATE_REQUEST")

    if not reject_reasons:
        if w_settled.value - w_amount.value < BUFFER_USD:
            hold_reasons.append("INSUFFICIENT_SETTLED_AFTER_BUFFER")
        if w_available.value - w_amount.value < BUFFER_USD:
            hold_reasons.append("INSUFFICIENT_AVAILABLE_AFTER_BUFFER")
        if w_days_since_change.value <= RECENT_DEST_DAYS:
            hold_reasons.append("DEST_CHANGED_RECENTLY")
        if w_speed.value == "urgent" and w_aml_tier.value in ("medium", "high"):
            hold_reasons.append("URGENT_RISK_TIER")

    if reject_reasons:
        decision = "REJECT"
        reasons  = reject_reasons
        severity = max(SEVERITY[r] for r in reasons)
        color    = "#e74c3c"
        icon     = "🚫"
    elif hold_reasons:
        decision = "HOLD"
        reasons  = hold_reasons
        severity = max(SEVERITY[r] for r in reasons)
        color    = "#f39c12"
        icon     = "⏸️"
    else:
        decision = "APPROVE"
        reasons  = ["NONE"]
        severity = 0
        color    = "#27ae60"
        icon     = "✅"

    with output:
        display(HTML(f"""
        <div style="
            border-left: 6px solid {color};
            background: #f9f9f9;
            padding: 16px 20px;
            border-radius: 6px;
            margin-top: 10px;
            font-family: monospace;
        ">
            <h3 style="color:{color}; margin:0 0 10px 0">{icon} Decision: {decision}</h3>
            <p style="margin:4px 0"><b>Reason Codes:</b> {", ".join(reasons)}</p>
            <p style="margin:4px 0"><b>Severity Score:</b> {severity}</p>
            <hr style="border:none; border-top:1px solid #ddd; margin:10px 0">
            <p style="margin:4px 0; color:#555; font-size:12px">
                Amount: ${w_amount.value:,.2f} &nbsp;|&nbsp;
                Available post-withdrawal: ${w_available.value - w_amount.value:,.2f} &nbsp;|&nbsp;
                Settled post-withdrawal: ${w_settled.value - w_amount.value:,.2f}
            </p>
        </div>
        """))

btn.on_click(run_simulation)

# ── Layout ───────────────────────────────────────────────────────
display(widgets.VBox([
    title,
    widgets.HBox([
        widgets.VBox([w_account_status, w_kyc_status, w_aml_tier, w_whitelisted]),
        widgets.VBox([w_amount, w_available, w_settled, w_speed]),
    ]),
    widgets.VBox([w_days_since_change, w_is_duplicate]),
    btn,
    output
]))

# Propuesta de Agentización

### El problema que resuelve

El motor actual es un script batch: procesa un archivo,
genera decisiones, exporta resultados. Funciona bien a pequeña escala,
pero en un entorno de producción con retiros llegando en tiempo real,
este modelo no es suficiente.

La agentización convierte el motor en un sistema autónomo,
reactivo y auditable.

---

### Flujo propuesto
```
[Solicitud de retiro entrante]
        ↓
[Event trigger — webhook o queue]
        ↓
[Agente de decisión — withdrawal_engine.py]
  ├── Consulta account_snapshot (base de datos en tiempo real)
  ├── Consulta destination_registry
  ├── Detecta duplicados en ventana de 15 min
  └── Aplica lógica REJECT → HOLD → APPROVE
        ↓
[Router de salida]
  ├── APPROVE → ejecuta retiro vía ACH/crypto API
  ├── HOLD    → crea ticket en cola de revisión manual
  └── REJECT  → notifica al cliente con reason code
        ↓
[Audit log — cada decisión queda registrada con timestamp]
```

---

### Stack tecnológico sugerido

| Componente           | Herramienta           | Razón                                        |
|----------------------|-----------------------|----------------------------------------------|
| Event queue          | AWS SQS / RabbitMQ    | Desacopla ingesta de procesamiento           |
| Motor de decisiones  | Python (este módulo)  | Ya construido, solo necesita API wrapper     |
| Base de datos        | PostgreSQL / DynamoDB | Snapshot de cuentas en tiempo real           |
| Orquestación         | Airflow / Prefect     | Para runs batch programados si aplica        |
| Cola de revisión     | Retool / Linear       | Interface para el equipo de operaciones      |
| Audit log            | S3 + Athena           | Trazabilidad completa, bajo costo            |
| Notificaciones       | Twilio / SendGrid     | Alertas a clientes y equipo interno          |
| Agente AI (opcional) | Claude API            | Revisión asistida de casos HOLD complejos    |

---

### Por qué este diseño es escalable

- **Stateless**: el motor no guarda estado entre decisiones,
  puede correr en paralelo sin conflictos
- **Parametrizable**: BUFFER_USD, RECENT_DEST_DAYS y DUP_WINDOW_MIN
  viven en un config file, no en el código
- **Auditable**: cada decisión queda registrada con su reason code,
  severidad y timestamp — cumplimiento regulatorio incluido
- **Extensible**: agregar una nueva regla es añadir un bloque
  al motor sin tocar el resto del sistema

### El rol del agente AI

En los casos HOLD, un agente basado en Claude podría analizar
el historial del cliente, el contexto de la cuenta y los reason codes
activados, y generar una recomendación para el analista de operaciones
antes de que este tome la decisión final. No reemplaza al humano —
lo hace más rápido y con más contexto.